In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import numpy as np
import json
import os

In [2]:
def get_embedding(text, embedding_model):
    """Generate an embedding for the given text."""
    return embedding_model.embed_query(text)

In [ ]:
# Initialize OpenAI LLM (compatible with langchain-core >= 1.0)
# Make sure you have OPENAI_API_KEY set in your environment or .openai_api_key file
api_key = None
if os.path.exists(".openai_api_key"):
    with open(".openai_api_key", "r") as f:
        api_key = f.read().strip()
elif "OPENAI_API_KEY" in os.environ:
    api_key = os.environ["OPENAI_API_KEY"]

llm = ChatOpenAI(
    model="gpt-4o-mini",  # Change to another model if needed (e.g., "gpt-4", "gpt-3.5-turbo")
    temperature=0.7,
    api_key=api_key
)

In [ ]:
# Define a prompt template (using ChatPromptTemplate for modern LangChain)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a medical AI assistant that helps expand concepts for a Concept Bottleneck Model."),
    ("human", """Given the disease "{disease}" and the existing concepts:
{existing_concepts}

Generate five new, distinct, and medically relevant concepts.""")
])

In [ ]:
# Updated prompt template for radiology and pulmonary diseases (using ChatPromptTemplate)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a highly knowledgeable medical AI assistant specializing in radiology and pulmonary diseases. Your task is to expand the list of medical concepts for a Concept Bottleneck Model that classifies diseases from chest X-ray images."),
    ("human", """Disease: "{disease}"
Existing Concepts:
{existing_concepts}

Please generate 10 additional, distinct, and medically relevant concepts related to this disease. 
Ensure they are specific to radiological findings, anatomical changes, and clinical correlations. Avoid duplication and generic terms.""")
])

In [ ]:
# Create a chain using the modern LangChain API (pipe operator)
llm_chain = prompt_template | llm

C:\Users\mehme\AppData\Local\Temp\ipykernel_1044\2720343761.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(llm=llm, prompt=prompt_template)


In [ ]:
def generate_new_concepts(disease, existing_concepts):
    """Generates and filters new concepts for a given disease."""
    # Use invoke instead of deprecated run method
    response = llm_chain.invoke({
        "disease": disease,
        "existing_concepts": "\n".join(existing_concepts)
    })
    
    # Extract content from AIMessage
    content = response.content if hasattr(response, 'content') else str(response)
    new_concepts = [line.strip() for line in content.split("\n") if line.strip()]
    
    return new_concepts

In [8]:
# Example Usage
disease = "Atelectasis"
existing_concepts = [
    "collapsed lung", "displacement of interlobar fissures", "reduced lung volume"
]

new_concepts = generate_new_concepts(disease, existing_concepts)
print("New Concepts for", disease, ":", new_concepts)  # Output newly generated concepts

C:\Users\mehme\AppData\Local\Temp\ipykernel_6972\1965649328.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm_chain.run({


New Concepts for Atelectasis : ['<think>', 'Alright, so I need to come up with 10 additional medical concepts related to atelectasis for a Concept Bottleneck Model in chest X-ray classification. Let me think through this step by step.', 'First, the existing concepts given are collapsed lung, displacement of interlobar fissures, and reduced lung volume. These make sense because atelectasis is when parts of the lung collapse, leading to these kinds of findings on imaging.', "I should focus on specific radiological findings or anatomical changes that are directly related to atelectasis. Maybe I can start by thinking about what happens when a portion of the lung collapses. The affected area might show signs like increased opacity, which could be called consolidative changes. That's one concept.", 'Next, the interlobar fissures displacement is already listed, so maybe other fissure-related findings. Perhaps the presence of linear opacities along the interlobar or intralobar regions. Wait, t

In [10]:
# Save new_concepts to JSON file
with open('data/concept_sets/xray-generated/atelactasis_new_concepts_v2.json', 'w') as f:
    json.dump(new_concepts, f, indent=4)

In [7]:
# Read initial concepts from JSON file
with open('data/concept_sets/gpt3_init/gpt35-turbo_chestxray_important_new.json', 'r') as f:
    initial_concepts = json.load(f)

# Process each disease and its concepts
for disease, existing_concepts in initial_concepts.items():
    print(f"Processing {disease}...")
    new_concepts = generate_new_concepts(disease, existing_concepts)
    
    # Store new concepts in a dictionary
    if 'new_concepts_dict' not in locals():
        new_concepts_dict = {}
    new_concepts_dict[disease] = new_concepts

# Save the generated concepts to a new JSON file
output_path = 'data/concept_sets/xray-generated/generated_concepts.json'
with open(output_path, 'w') as f:
    json.dump(new_concepts_dict, f, indent=4)

print(f"Generated concepts saved to {output_path}")

Processing Atelectasis...


C:\Users\mehme\AppData\Local\Temp\ipykernel_1044\1965649328.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = llm_chain.run({


Processing Consolidation...
Processing Infiltration...
Processing Pneumothorax...
Processing Edema...
Processing Emphysema...
Processing Fibrosis...
Processing Effusion...
Processing Pneumonia...
Processing Pleural_thickening...
Processing Cardiomegaly...
Processing Nodule Mass...
Processing Hernia...
Generated concepts saved to data/concept_sets/xray-generated/generated_concepts.json
